# Laboratorio — Robot de entregas en un almacén

A partir de la **imagen**, construye el MDP y resuélvelo con **Value Iteration** y **Policy Iteration**.

![Mundo del ejercicio](https://drive.google.com/uc?export=view&id=1_sJaD57gHuiz1joEgl4B-u0aDy8jtMDo)



## Convención y notación

$$
s=(row,col)
$$

$$
T(s,a,s')=P(s'\mid s,a)
$$

$$
R(s)
$$

Para Value Iteration:

$$
V_{k+1}(s)
=
R(s)
+
\gamma
\max_a
\sum_{s'}T(s,a,s')V_k(s')
$$

Para Policy Evaluation:

$$
V_{k+1}^{\pi}(s)
=
R(s)
+
\gamma
\sum_{s'}T(s,\pi(s),s')V_k^\pi(s')
$$

### Acciones

```python
UP    = (-1, 0)
DOWN  = ( 1, 0)
LEFT  = ( 0,-1)
RIGHT = ( 0, 1)
```



## Reglas del mundo

El grid tiene **5 filas × 6 columnas**.

### Estados especiales

A partir de la imagen identifica:

- `START`
- estanterías / paredes;
- zona de entrega `+10` (**terminal**);
- estación de carga `+2` (**terminal**);
- peligro mortal `-10` (**terminal**);
- peligros `-3` (**no terminales**);
- celdas de piso resbaloso.

### Recompensa

Usamos la convención del notebook de clase, es decir, **\(R(s)\)**:

- entrega: `+10`;
- carga: `+2`;
- peligro mortal: `-10`;
- peligro: `-3`;
- cualquier otro estado transitable: `-1` (costo por paso).

### Dinámica

La transición depende del **estado actual**:

**Piso normal**

$$
P(\text{dirección elegida})=0.90
$$

$$
P(\text{desviación izquierda})=0.05
$$

$$
P(\text{desviación derecha})=0.05
$$

**Piso resbaloso**

$$
P(\text{dirección elegida})=0.60
$$

$$
P(\text{desviación izquierda})=0.20
$$

$$
P(\text{desviación derecha})=0.20
$$

Si el movimiento sale del grid o golpea una estantería, el robot **permanece en el mismo estado**.

Usa:

$$
\gamma=0.9,\qquad \theta=10^{-4}
$$



## Parte 1 — Modela el MDP

Completa la clase `WarehouseMDP`.

La parte importante no es escribir muchas líneas de código: es traducir correctamente la imagen a:

- estados;
- acciones;
- recompensas;
- terminales;
- obstáculos;
- tipos de piso;
- función de transición.


In [1]:
import numpy as np

class WarehouseMDP:
    def __init__(self):
        self.height = 5
        self.width = 6

        self.start = (0, 0)
        self.walls = {(0, 3), (1, 1), (2, 4), (4, 2)}
        self.slippery_states = {(1, 2), (2, 1), (3, 3)}

        self.terminal_states = {
            (0, 5): 10.0,
            (2, 2): 2.0,
            (3, 5): -10.0,
        }

        self.danger_states = {
            (1, 4): -3.0,
            (4, 1): -3.0,
        }

        self.living_reward = -1.0
        self.gamma = 0.9

        self.actions = [
            (-1, 0),
            ( 1, 0),
            ( 0,-1),
            ( 0, 1),
        ]

        self.p_normal = (0.90, 0.05, 0.05)
        self.p_slippery = (0.60, 0.20, 0.20)

        assert self.start is not None and self.terminal_states, (
            "Falta completar el layout (start, walls, slippery_states, "
            "terminal_states, danger_states) a partir de la imagen."
        )

    def is_valid_state(self, state):
        row, col = state
        inside = 0 <= row < self.height and 0 <= col < self.width
        return inside and state not in self.walls

    def states(self):
        return [
            (r, c)
            for r in range(self.height)
            for c in range(self.width)
            if (r, c) not in self.walls
        ]

    def is_terminal(self, state):
        return state in self.terminal_states

    def get_reward(self, state):
        if self.is_terminal(state):
            return self.terminal_states[state]
        if state in self.danger_states:
            return self.danger_states[state]
        return self.living_reward

    def get_transition_probs(self, state, action):
        """
        Devuelve:
            [(next_state, probability), ...]

        - las probabilidades dependen de si 'state' es resbaloso;
        - si golpea pared/borde, next_state = state.
        """
        if self.is_terminal(state):
            return [(state, 1.0)]

        p_intended, p_left, p_right = (
            self.p_slippery if state in self.slippery_states else self.p_normal
        )

        left = (-action[1], -action[0])
        right = (action[1], action[0])

        outcomes = [(action, p_intended), (left, p_left), (right, p_right)]

        probs = {}
        for move, p in outcomes:
            next_state = (state[0] + move[0], state[1] + move[1])
            if not self.is_valid_state(next_state):
                next_state = state
            probs[next_state] = probs.get(next_state, 0.0) + p

        return list(probs.items())


### Validación mínima del modelo

Antes de implementar Bellman, valida primero el MDP.


In [2]:
grid = WarehouseMDP()

S = grid.states()
print("Número de estados:", len(S))

for s in S:
    for a in grid.actions:
        transitions = grid.get_transition_probs(s, a)
        total = sum(p for _, p in transitions)
        assert abs(total - 1.0) < 1e-12

print("✓ Todas las distribuciones de transición suman 1.")

Número de estados: 26
✓ Todas las distribuciones de transición suman 1.



## Parte 2 — Value Iteration

Implementa:

$$
V_{k+1}(s)
=
R(s)+\gamma\max_a
\sum_{s'}T(s,a,s')V_k(s')
$$


In [3]:
def expected_next_value(grid, state, action, V):
    return sum(
        prob * V[next_state]
        for next_state, prob in grid.get_transition_probs(state, action)
    )

def value_iteration(grid, threshold=1e-4, max_iter=10_000):
    V = {s: 0.0 for s in grid.states()}

    for iteration in range(max_iter):
        V_new = {}
        delta = 0.0

        for s in grid.states():
            if grid.is_terminal(s):
                V_new[s] = grid.get_reward(s)
            else:
                best = max(
                    expected_next_value(grid, s, a, V) for a in grid.actions
                )
                V_new[s] = grid.get_reward(s) + grid.gamma * best

            delta = max(delta, abs(V_new[s] - V[s]))

        V = V_new

        if delta < threshold:
            break

    return V, iteration + 1

def extract_policy(grid, V):
    policy = {}
    for s in grid.states():
        if grid.is_terminal(s):
            continue
        policy[s] = max(
            grid.actions,
            key=lambda a: expected_next_value(grid, s, a, V)
        )
    return policy


## Parte 3 — Policy Iteration

### Policy Evaluation

$$
V_{k+1}^{\pi}(s)
=
R(s)+\gamma
\sum_{s'}T(s,\pi(s),s')V_k^\pi(s')
$$

### Policy Improvement

$$
\pi_{\mathrm{new}}(s)
=
\arg\max_a
\sum_{s'}T(s,a,s')V^\pi(s')
$$

In [4]:
def policy_evaluation(grid, policy, threshold=1e-4, max_iter=10_000):
    V = {s: 0.0 for s in grid.states()}

    for iteration in range(max_iter):
        V_new = {}
        delta = 0.0

        for s in grid.states():
            if grid.is_terminal(s):
                V_new[s] = grid.get_reward(s)
            else:
                a = policy[s]
                V_new[s] = (
                    grid.get_reward(s)
                    + grid.gamma * expected_next_value(grid, s, a, V)
                )
            delta = max(delta, abs(V_new[s] - V[s]))

        V = V_new

        if delta < threshold:
            break

    return V

def policy_improvement(grid, V):
    return extract_policy(grid, V)

def policy_iteration(grid, threshold=1e-4, max_iter=100):
    policy = {
        s: grid.actions[0]
        for s in grid.states()
        if not grid.is_terminal(s)
    }

    history = []

    for iteration in range(max_iter):
        V = policy_evaluation(grid, policy, threshold=threshold)

        new_policy = policy_improvement(grid, V)

        changed = sum(new_policy[s] != policy[s] for s in policy)
        history.append(changed)
        policy = new_policy

        if changed == 0:
            break

    V = policy_evaluation(grid, policy, threshold=threshold)
    return policy, V, history


## Parte 4 — Visualización y comparación


In [5]:
ARROWS = {
    (-1, 0): "↑",
    ( 1, 0): "↓",
    ( 0,-1): "←",
    ( 0, 1): "→",
}

def print_values(grid, V):
    for r in range(grid.height):
        row = []
        for c in range(grid.width):
            s = (r, c)
            if s in grid.walls:
                row.append("  WALL  ")
            else:
                row.append(f"{V[s]:+7.3f}")
        print(" | ".join(row))

def print_policy(grid, policy):
    for r in range(grid.height):
        row = []
        for c in range(grid.width):
            s = (r, c)

            if s in grid.walls:
                row.append(" # ")
            elif grid.is_terminal(s):
                reward = grid.get_reward(s)
                row.append(f"{reward:+.0f}")
            else:
                row.append(f" {ARROWS[policy[s]]} ")

        print(" | ".join(row))

In [6]:
V_vi, n_vi = value_iteration(grid)
pi_vi = extract_policy(grid, V_vi)

print("=== VALUE ITERATION ===")
print("Iteraciones:", n_vi)
print("\nValores:")
print_values(grid, V_vi)
print("\nPolítica:")
print_policy(grid, pi_vi)

pi_pi, V_pi, history = policy_iteration(grid)

print("\n=== POLICY ITERATION ===")
print("Historia:", history)
print("\nValores:")
print_values(grid, V_pi)
print("\nPolítica:")
print_policy(grid, pi_pi)

assert pi_vi == pi_pi
print("\n✓ Ambos algoritmos encontraron la misma política óptima.")

=== VALUE ITERATION ===
Iteraciones: 20

Valores:
 -2.575 |  -1.679 |  -0.652 |   WALL   |  +7.607 | +10.000
 -2.193 |   WALL   |  +0.560 |  +2.104 |  +3.670 |  +7.607
 -1.229 |  -0.063 |  +2.000 |  +0.832 |   WALL   |  +5.673
 -1.770 |  -0.731 |  +0.552 |  -0.782 |  -1.837 | -10.000
 -2.732 |  -3.890 |   WALL   |  -1.837 |  -2.692 |  -3.802

Política:
 →  |  →  |  ↓  |  #  |  →  | +10
 ↓  |  #  |  ↓  |  →  |  →  |  ↑ 
 →  |  →  | +2 |  ↑  |  #  |  ↑ 
 →  |  →  |  ↑  |  ↑  |  ←  | -10
 ↑  |  ↑  |  #  |  ↑  |  ←  |  ← 

=== POLICY ITERATION ===
Historia: [16, 5, 1, 0]

Valores:
 -2.575 |  -1.679 |  -0.652 |   WALL   |  +7.607 | +10.000
 -2.193 |   WALL   |  +0.560 |  +2.104 |  +3.670 |  +7.607
 -1.229 |  -0.063 |  +2.000 |  +0.832 |   WALL   |  +5.673
 -1.770 |  -0.731 |  +0.552 |  -0.782 |  -1.837 | -10.000
 -2.732 |  -3.890 |   WALL   |  -1.837 |  -2.692 |  -3.802

Política:
 →  |  →  |  ↓  |  #  |  →  | +10
 ↓  |  #  |  ↓  |  →  |  →  |  ↑ 
 →  |  →  | +2 |  ↑  |  #  |  ↑ 
 →  |  →  


## Parte 5 — Interpreta la política

Antes de cambiar parámetros, responde:

1. Desde `START`, ¿el robot busca la **entrega +10** o prefiere la **estación de carga +2**?
2. ¿Por qué una recompensa menor podría ser óptima?
3. ¿En qué estados el piso resbaloso cambia la decisión?
4. ¿Qué papel cumple el costo por paso `-1`?
5. ¿Por qué \(T(s,a,s')\) ya no puede implementarse con las mismas probabilidades para todos los estados?

### Experimento A — Menos costo por paso

Cambia:

```python
living_reward = -0.1
```

Predice la política **antes de ejecutar**.

### Experimento B — Piso muy resbaloso

Cambia la probabilidad de movimiento deseado del piso resbaloso:

```python
0.60 → 0.40
```

y reparte el restante entre las dos desviaciones.

### Experimento C — Más paciencia

Cambia:

```python
gamma = 0.99
```

¿La política valora más la recompensa `+10` distante?

### Bonus

Encuentra aproximadamente el valor de `living_reward` a partir del cual la política desde `START` cambia entre:

- ir a carga `+2`;
- intentar llegar a entrega `+10`.


### Resultados con el mundo base

In [7]:
def destino_final(grid, policy, pasos=2000):
    dist = {grid.start: 1.0}
    final = {}
    for _ in range(pasos):
        nueva = {}
        for s, p in dist.items():
            if grid.is_terminal(s):
                final[s] = final.get(s, 0.0) + p
                continue
            for s2, q in grid.get_transition_probs(s, policy[s]):
                nueva[s2] = nueva.get(s2, 0.0) + p * q
        dist = nueva
        if sum(dist.values()) < 1e-9:
            break
    return final

def resolver(living_reward=-1.0, gamma=0.9, p_slippery=(0.6, 0.2, 0.2), sin_resbaloso=False):
    g = WarehouseMDP()
    g.living_reward = living_reward
    g.gamma = gamma
    g.p_slippery = p_slippery
    if sin_resbaloso:
        g.slippery_states = set()
    V, n_iter = value_iteration(g)
    return g, V, extract_policy(g, V), n_iter

def reporte(titulo, **kwargs):
    g, V, pol, n_iter = resolver(**kwargs)
    final = destino_final(g, pol)
    print(f"=== {titulo} ===")
    print(f"Iteraciones: {n_iter} | V(START) = {V[g.start]:.3f}")
    print(f"Entrega (+10): {final.get((0, 5), 0):.1%} | Carga (+2): {final.get((2, 2), 0):.1%} | Peligro (-10): {final.get((3, 5), 0):.1%}")
    print_policy(g, pol)
    print()
    return g, V, pol

g_base, V_base, pol_base = reporte("Base (living = -1, gamma = 0.9)")
g_ns, V_ns, pol_ns = reporte("Mismo mundo sin piso resbaloso", sin_resbaloso=True)

cambian = [s for s in pol_base if pol_base[s] != pol_ns[s]]
print("Estados donde cambia la decisión por el piso resbaloso:", cambian)

=== Base (living = -1, gamma = 0.9) ===
Iteraciones: 20 | V(START) = -2.575
Entrega (+10): 23.7% | Carga (+2): 76.3% | Peligro (-10): 0.0%
 →  |  →  |  ↓  |  #  |  →  | +10
 ↓  |  #  |  ↓  |  →  |  →  |  ↑ 
 →  |  →  | +2 |  ↑  |  #  |  ↑ 
 →  |  →  |  ↑  |  ↑  |  ←  | -10
 ↑  |  ↑  |  #  |  ↑  |  ←  |  ← 

=== Mismo mundo sin piso resbaloso ===
Iteraciones: 19 | V(START) = -2.407
Entrega (+10): 89.5% | Carga (+2): 10.5% | Peligro (-10): 0.0%
 →  |  →  |  ↓  |  #  |  →  | +10
 ↓  |  #  |  →  |  →  |  →  |  ↑ 
 →  |  →  | +2 |  ↑  |  #  |  ↑ 
 →  |  ↑  |  ↑  |  ↑  |  ←  | -10
 ↑  |  ↑  |  #  |  ↑  |  ←  |  ← 

Estados donde cambia la decisión por el piso resbaloso: [(1, 2), (3, 1)]


### Experimentos A, B y C

In [8]:
reporte("Experimento A: living_reward = -0.1", living_reward=-0.1)
reporte("Experimento B: piso muy resbaloso (0.4 / 0.3 / 0.3)", p_slippery=(0.4, 0.3, 0.3))
resultado_c = reporte("Experimento C: gamma = 0.99", gamma=0.99)

=== Experimento A: living_reward = -0.1 ===
Iteraciones: 26 | V(START) = 1.649
Entrega (+10): 74.8% | Carga (+2): 25.2% | Peligro (-10): 0.0%
 →  |  →  |  ↓  |  #  |  →  | +10
 ↑  |  #  |  →  |  →  |  →  |  ↑ 
 →  |  →  | +2 |  ↑  |  #  |  ↑ 
 →  |  →  |  →  |  ↑  |  ←  | -10
 ↑  |  ↑  |  #  |  ↑  |  ←  |  ← 

=== Experimento B: piso muy resbaloso (0.4 / 0.3 / 0.3) ===
Iteraciones: 22 | V(START) = -2.706
Entrega (+10): 40.6% | Carga (+2): 59.4% | Peligro (-10): 0.0%
 →  |  →  |  ↓  |  #  |  →  | +10
 ↓  |  #  |  ↓  |  →  |  →  |  ↑ 
 →  |  →  | +2 |  ↑  |  #  |  ↑ 
 →  |  →  |  ↑  |  ↑  |  ←  | -10
 ↑  |  ↑  |  #  |  ↑  |  ←  |  ← 

=== Experimento C: gamma = 0.99 ===
Iteraciones: 24 | V(START) = -1.456
Entrega (+10): 70.9% | Carga (+2): 29.1% | Peligro (-10): 0.0%
 →  |  →  |  ↓  |  #  |  →  | +10
 ↓  |  #  |  →  |  →  |  →  |  ↑ 
 →  |  →  | +2 |  ↑  |  #  |  ↑ 
 →  |  →  |  ↑  |  ↑  |  ←  | -10
 ↑  |  ↑  |  #  |  ↑  |  ←  |  ← 



### Bonus — ¿desde qué costo por paso cambia la política?

In [9]:
import numpy as np

anterior = None
for lr in np.arange(-1.2, -0.3, 0.005):
    g, V, pol, _ = resolver(living_reward=float(lr))
    accion = ARROWS[pol[(1, 2)]]
    if accion != anterior:
        final = destino_final(g, pol)
        print(f"living_reward = {lr:+.3f} -> acción en (1,2): {accion} | entrega {final.get((0, 5), 0):.1%} | carga {final.get((2, 2), 0):.1%}")
    anterior = accion

living_reward = -1.200 -> acción en (1,2): ↓ | entrega 22.5% | carga 77.5%


living_reward = -0.795 -> acción en (1,2): → | entrega 70.9% | carga 29.1%


### Respuestas — Parte 5

**1. Desde START, ¿busca la entrega o la carga?**
Con el costo por paso de -1, el robot busca la **estación de carga (+2)**. La política manda al robot derecha, derecha, abajo y abajo (START → (0,1) → (0,2) → (1,2) → (2,2)). Siguiendo esa política termina en la carga el 76.3 % de las veces y en la entrega el 23.7 % (por los resbalones, que lo desvían hacia el pasillo de la derecha).

**2. ¿Por qué una recompensa menor podría ser óptima?**
Porque la recompensa +10 queda lejos y es cara de alcanzar. Las estanterías en (0,3) y (2,4) y el peligro mortal en (3,5) cierran los demás caminos, así que para llegar a la entrega hay que pasar sí o sí por la celda de peligro (1,4) de -3. Son 7 pasos de -1 más ese -3, además del riesgo de resbalar. La carga queda a solo 4 pasos y sin peligros en el camino. Con ese costo, llegar rápido a +2 vale más (V(START) = -2.575) que recorrer todo el camino hasta +10.

**3. ¿En qué estados el piso resbaloso cambia la decisión?**
Para comparar, resolví el mismo mundo con todo el piso normal. La decisión cambia en dos estados: **(1,2)** y **(3,1)**. En (1,2), que es resbaloso, con piso normal el robot seguiría a la derecha hacia la entrega, pero con piso resbaloso prefiere bajar directo a la carga, que está justo debajo. En (3,1) con piso resbaloso prefiere ir a la derecha en vez de subir, probablemente para no entrar a la celda resbalosa (2,1) que queda arriba. El efecto total es grande: sin resbalar, el robot terminaría en la entrega el 89.5 % de las veces; con el piso resbaloso solo el 23.7 %.

**4. ¿Qué papel cumple el costo por paso -1?**
Hace que cada paso extra cueste, así que el robot prefiere rutas cortas aunque la recompensa final sea menor. Mientras más caro es moverse, menos conviene ir a una recompensa lejana, como se ve en el experimento A.

**5. ¿Por qué T(s,a,s') no se puede implementar igual para todos los estados?**
Porque la probabilidad de moverse como se quiere depende del estado actual: en piso normal es 0.90/0.05/0.05 y en piso resbaloso 0.60/0.20/0.20. Por eso `get_transition_probs` tiene que revisar si `state` está en `slippery_states` antes de armar las probabilidades.

### Experimentos

**A. living_reward = -0.1.** Esperaba que con el movimiento más barato valiera la pena la ruta larga. Pasó eso: la política cambia en tres estados ((1,0), (1,2) y (3,2)) y desde START el robot ya va hacia la entrega, que alcanza el 74.8 % de las veces (antes 23.7 %). El valor de START sube de -2.575 a +1.649.

**B. Piso muy resbaloso (0.4 / 0.3 / 0.3).** La política no cambió. Lo que cambió fue el resultado esperado: el valor de START baja un poco (de -2.575 a -2.706) y como las desviaciones son más frecuentes, el robot termina en la entrega el 40.6 % de las veces en lugar del 23.7 %, aunque siga intentando ir a la carga.

**C. gamma = 0.99.** Sí, con más paciencia la política valora más el +10 lejano: cambia la decisión en (1,2), que pasa de bajar a la carga a ir hacia la derecha. Desde START el robot llega a la entrega el 70.9 % de las veces y el valor de START sube a -1.456.

**Bonus.** Probando valores de `living_reward`, la decisión en (1,2) pasa de ir a la carga a ir hacia la entrega alrededor de **-0.8** (cambia entre -0.800 y -0.795). Con un costo por paso más negativo que eso conviene la carga y con uno menos negativo conviene la entrega.

---
### Declaración de uso de IA generativa

Herramienta: Claude (Anthropic). Propósito: asistencia en programación (clase `WarehouseMDP`, Value Iteration, Policy Iteration y los experimentos) y borrador de las respuestas de la Parte 5. El layout lo tomé de la imagen del enunciado y las respuestas las revisé contra los resultados que salen en el notebook.